#Library

In [ ]:
import pandas as pd
import numpy as np
from transformers import RobertaTokenizerFast, RobertaForTokenClassification, Trainer, TrainingArguments, AutoTokenizer, DataCollatorForTokenClassification
from peft import get_peft_model, LoraConfig, TaskType
import torch
from torch.utils.data import Dataset
import time
import os
import json
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.metrics.sequence_labeling import get_entities


#LOAD DATA

In [ ]:
class NERDataset(Dataset):
    """
    A PyTorch Dataset wrapper for tokenized NER data.
    """
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

def load_data(file_path):
    """
    Load BIO data from a JSON or JSONL file.
    """
    data = []
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().strip()
            if content.startswith("["):
                data = json.loads(content)
            else:
                for line in content.splitlines():
                    if line.strip():
                        data.append(json.loads(line))
        print(f"Successfully loaded {len(data)} examples from {file_path}")
    except Exception as e:
        print(f"Error loading file {file_path}: {e}")
        if file_path.endswith(".csv"):
            try:
                df = pd.read_csv(file_path)
                for _, row in df.iterrows():
                    tokens = eval(row["tokens"]) if isinstance(row["tokens"], str) and row["tokens"].startswith("[") else str(row["tokens"]).split()
                    tags = eval(row["tags"]) if isinstance(row["tags"], str) and row["tags"].startswith("[") else str(row["tags"]).split()
                    data.append({"tokens": tokens, "tags": tags})
                print(f"Successfully loaded {len(data)} examples from CSV: {file_path}")
            except Exception as csv_err:
                print(f"Failed loading as CSV: {csv_err}")
    return data

def get_tag_mappings(data):
    """
    Extract unique tags from data and create mappings to/from integer IDs.
    """
    unique_tags = set()
    for item in data:
        unique_tags.update(item["tags"])
    
    sorted_tags = sorted(list(unique_tags))
    if "O" in sorted_tags:
        sorted_tags.remove("O")
        sorted_tags = ["O"] + sorted_tags
        
    tag2id = {tag: i for i, tag in enumerate(sorted_tags)}
    id2tag = {i: tag for i, tag in enumerate(sorted_tags)}
    return tag2id, id2tag

def preprocess_and_tokenize(data, tokenizer: RobertaTokenizerFast, tag2id):
    """
    Preprocess raw text and tokenize into token-level inputs aligned with BIO labels.
    """
    texts = [x["tokens"] for x in data]
    tags = [x["tags"] for x in data]
    
    tokenized_inputs = tokenizer(
        texts,
        is_split_into_words=True,
        truncation=True,
        padding=True
    )
    
    labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(tag2id.get(label[word_idx], tag2id.get("O", 0)))
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


#evaluate

In [ ]:
def evaluate_linguistic_metrics(predictions, references):
    """
    Calculate Precision, Recall, and F1-Score (Macro-average).
    """
    report = classification_report(references, predictions)
    f1 = f1_score(references, predictions)
    return report, f1

def profile_computing_performance(model_fn, *args, **kwargs):
    """
    Measure peak GPU memory usage (VRAM) in GB and execution time in seconds.
    """
    use_cuda = torch.cuda.is_available()
    if use_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    start_time = time.time()
    result = model_fn(*args, **kwargs)
    execution_time = time.time() - start_time
    
    peak_vram = 0.0
    if use_cuda:
        peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
        
    return result, execution_time, peak_vram

def evaluate_muc5_errors(predictions, references):
    """
    Adapt MUC-5 evaluation guidelines under exact boundary match.
    """
    cor, inc, mis, spu = 0, 0, 0, 0
    for pred_seq, ref_seq in zip(predictions, references):
        ref_ents = get_entities(ref_seq)
        pred_ents = get_entities(pred_seq)
        ref_by_boundary = {(start, end): ent_type for ent_type, start, end in ref_ents}
        pred_by_boundary = {(start, end): ent_type for ent_type, start, end in pred_ents}
        for boundary, ref_type in ref_by_boundary.items():
            if boundary in pred_by_boundary:
                pred_type = pred_by_boundary[boundary]
                if ref_type == pred_type:
                    cor += 1
                else:
                    inc += 1
            else:
                mis += 1
        for boundary in pred_by_boundary:
            if boundary not in ref_by_boundary:
                spu += 1
    return {"COR": cor, "INC": inc, "MIS": mis, "SPU": spu}

def evaluate_fine_grained(predictions, references, sentences=None):
    """
    Fine-grained analysis based on entity properties (eLen, eCon, eFre).
    """
    from collections import Counter
    ref_entities_list = []
    entity_text_labels = {}
    entity_text_freq = Counter()
    
    for sent_idx, ref_seq in enumerate(references):
        ref_ents = get_entities(ref_seq)
        for ent_type, start, end in ref_ents:
            if sentences and sent_idx < len(sentences):
                text = " ".join(sentences[sent_idx][start:end]).lower()
            else:
                text = f"entity_{start}_{end}"
            ref_entities_list.append((sent_idx, start, end, ent_type, text))
            entity_text_freq[text] += 1
            if text not in entity_text_labels:
                entity_text_labels[text] = []
            entity_text_labels[text].append(ent_type)
            
    entity_consistency = {}
    for text, labels in entity_text_labels.items():
        most_common_count = Counter(labels).most_common(1)[0][1]
        entity_consistency[text] = most_common_count / len(labels)
        
    pred_entities_by_sent = []
    for pred_seq in predictions:
        pred_ents = get_entities(pred_seq)
        pred_entities_by_sent.append({(start, end): ent_type for ent_type, start, end in pred_ents})
        
    categories = {
        "eLen_short": {"total": 0, "correct": 0},
        "eLen_long": {"total": 0, "correct": 0},
        "eCon_consistent": {"total": 0, "correct": 0},
        "eCon_inconsistent": {"total": 0, "correct": 0},
        "eFre_few_shot": {"total": 0, "correct": 0},
        "eFre_many_shot": {"total": 0, "correct": 0}
    }
    
    for sent_idx, start, end, ref_type, text in ref_entities_list:
        correct = False
        if sent_idx < len(pred_entities_by_sent):
            pred_sent = pred_entities_by_sent[sent_idx]
            if (start, end) in pred_sent and pred_sent[(start, end)] == ref_type:
                correct = True
        length = end - start
        len_cat = "eLen_long" if length >= 4 else "eLen_short"
        categories[len_cat]["total"] += 1
        if correct:
            categories[len_cat]["correct"] += 1
        con = entity_consistency[text]
        con_cat = "eCon_consistent" if con == 1.0 else "eCon_inconsistent"
        categories[con_cat]["total"] += 1
        if correct:
            categories[con_cat]["correct"] += 1
        freq = entity_text_freq[text]
        fre_cat = "eFre_few_shot" if freq <= 2 else "eFre_many_shot"
        categories[fre_cat]["total"] += 1
        if correct:
            categories[fre_cat]["correct"] += 1
            
    results = {}
    for cat, stats in categories.items():
        total = stats["total"]
        correct = stats["correct"]
        acc = correct / total if total > 0 else 0.0
        results[cat] = {"total": total, "correct": correct, "accuracy": acc}
    return results


##model

In [ ]:
def get_roberta_lora_model(model_name="roberta-base", num_labels=19, r=8, lora_alpha=16, lora_dropout=0.1):
    """
    Load pre-trained RoBERTa model for token classification, freeze base weights,
    and inject LoRA parameters.
    """
    model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=num_labels)
    peft_config = LoraConfig(
        task_type=TaskType.TOKEN_CLS,
        inference_mode=False,
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=["query", "value"]
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    return model


train

In [ ]:
def get_compute_metrics_fn(id2tag):
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)
        true_predictions = [
            [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        true_labels = [
            [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]
        return {
            "precision": precision_score(true_labels, true_predictions),
            "recall": recall_score(true_labels, true_predictions),
            "f1": f1_score(true_labels, true_predictions)
        }
    return compute_metrics

def train_model(model, train_dataset, eval_dataset, training_args, id2tag, data_collator=None):
    compute_metrics_fn = get_compute_metrics_fn(id2tag)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn
    )
    trainer.train()
    return trainer

def grid_search_lora(train_dataset, eval_dataset, id2tag, model_name="roberta-base", num_labels=19, output_dir="./results"):
    r_values = [4, 8, 16]
    alpha_values = [8, 16, 32]
    results = []
    for r in r_values:
        for alpha in alpha_values:
            print(f"\nTraining LoRA model with rank={r}, alpha={alpha}")
            exp_dir = os.path.join(output_dir, f"lora_r{r}_a{alpha}")
            os.makedirs(exp_dir, exist_ok=True)
            training_args = TrainingArguments(
                output_dir=exp_dir,
                eval_strategy="epoch",
                save_strategy="epoch",
                learning_rate=5e-4,
                per_device_train_batch_size=8,
                per_device_eval_batch_size=8,
                num_train_epochs=3,
                weight_decay=0.01,
                logging_steps=10,
                load_best_model_at_end=True,
                metric_for_best_model="f1",
                report_to="none"
            )
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            data_collator = DataCollatorForTokenClassification(tokenizer)
            def init_and_train():
                model = get_roberta_lora_model(model_name=model_name, num_labels=num_labels, r=r, lora_alpha=alpha)
                trainer = train_model(model=model, train_dataset=train_dataset, eval_dataset=eval_dataset, training_args=training_args, id2tag=id2tag, data_collator=data_collator)
                return trainer, model
            (trainer, model), training_time, peak_vram = profile_computing_performance(init_and_train)
            print(f"Finished in {training_time:.2f}s. Peak VRAM: {peak_vram:.4f} GB")
            eval_metrics = trainer.evaluate()
            results.append({
                "rank": r,
                "alpha": alpha,
                "f1": eval_metrics.get("eval_f1", 0.0),
                "precision": eval_metrics.get("eval_precision", 0.0),
                "recall": eval_metrics.get("eval_recall", 0.0),
                "training_time_sec": training_time,
                "peak_vram_gb": peak_vram
            })
            if torch.cuda.is_available():
                del model
                del trainer
                torch.cuda.empty_cache()
    df_results = pd.DataFrame(results)
    results_path = os.path.join(output_dir, "grid_search_results.csv")
    df_results.to_csv(results_path, index=False)
    print(f"\nSaved grid search to {results_path}")
    return df_results
